In [ ]:
from seqeval.metrics.sequence_labeling import get_entities


def merge_tokens_and_tags(tokens, tags):
    """
    Merge tokens according to BIO tags using seqeval.
    Returns:
      merged_tokens, merged_tags
    """
    merged_tokens = []
    merged_tags = []

    # Extract entities as (type, start_idx, end_idx)
    entities = get_entities(tags)

    idx = 0
    for ent_type, start, end in entities:
        # Add tokens before entity (with tag 'O')
        while idx < start:
            merged_tokens.append(tokens[idx])
            merged_tags.append("O")
            idx += 1

        # Merge tokens that belong to the entity
        entity_text = " ".join(tokens[start : end + 1])
        merged_tokens.append(entity_text)
        merged_tags.append(ent_type)
        idx = end + 1

    # Add remaining tokens
    while idx < len(tokens):
        merged_tokens.append(tokens[idx])
        merged_tags.append("O")
        idx += 1

    return merged_tokens, merged_tags

In [ ]:
from itertools import groupby
from datasets import Dataset, DatasetDict

conll_dir = "../data/conll2003/"


def build_conll_ds(lines: str) -> list[dict]:
    rows = []
    results = [list(group) for k, group in groupby(lines, lambda x: x == "\n") if not k]
    for i, r in enumerate(results[1:]):
        tokens, pos_tags, chunk_tags, ner_tags = [], [], [], []
        for col in r:
            col = col.replace("\n", "").split(" ")
            sentence_id = i
            tokens.append(col[0])
            pos_tags.append(col[1])
            chunk_tags.append(col[2])
            ner_tags.append(col[3])

        rows.append(
            {
                "sentence_id": sentence_id,
                "tokens": tokens,
                "pos_tags": pos_tags,
                "chunk_tags": chunk_tags,
                "ner_tags": ner_tags,
            }
        )

    return rows


with open(f"{conll_dir}/train.txt") as train_file:
    train_rows = train_file.readlines()

with open(f"{conll_dir}/test.txt") as test_file:
    test_rows = test_file.readlines()

with open(f"{conll_dir}/valid.txt") as val_file:
    val_rows = val_file.readlines()

train = Dataset.from_list(build_conll_ds(train_rows))
test = Dataset.from_list(build_conll_ds(test_rows))
val = Dataset.from_list(build_conll_ds(val_rows))

dataset = DatasetDict(
    {
        "train": train,
        "validation": test,
        "test": val,
    }
)

In [ ]:
import json

output = []
for i, line in enumerate(dataset["validation"]):
    item = {}
    item["id"] = i
    item["input"] = " ".join(line["tokens"])

    # Merge tokens and tags
    tokens, tags = merge_tokens_and_tags(line["tokens"], line["ner_tags"])

    new_line = ""
    for tok, tag in zip(tokens, tags):
        if tag == "O":
            new_line += tok + " "
        elif tag == "PER":
            new_line += "((" + tok + ")) "
        elif tag == "LOC":
            new_line += "[[" + tok + "]] "
        elif tag == "ORG":
            new_line += "{{" + tok + "}} "
        elif tag == "MISC":
            new_line += "<<" + tok + ">> "

    item["text"] = new_line

    output.append(item)

with open(conll_dir + "validation_parallel.json", "w") as f:
    (json.dump(output, f))

In [ ]:
import json
from sklearn.model_selection import train_test_split

# Load jsonl file into a list of dictionaries
with open("../data/labelled/all_parallel.jsonl") as test_f:
    data = [json.loads(line) for line in test_f]

# Create test split (notice: fixed seed)
train_data, test_data = train_test_split(data, test_size=0.3, random_state=52)